In [ ]:
# =========================================================
# 🍽️  RECIPE RECOMMENDATION SYSTEM
# Topics: TF-IDF, Cosine Similarity, Euclidean Distance,
#         KNN, PCA, KMeans
# =========================================================

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing           import OneHotEncoder, MinMaxScaler, LabelEncoder
from sklearn.metrics.pairwise        import cosine_similarity, euclidean_distances
from sklearn.neighbors               import KNeighborsClassifier
from sklearn.decomposition           import PCA
from sklearn.cluster                 import KMeans
from sklearn.model_selection         import cross_val_score
from scipy.sparse                    import hstack, csr_matrix


# =========================================================
# 1. LOAD DATA
# =========================================================
def load_data(filepath="ml_recipes.csv"):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Dataset not found: {filepath}")

    df = pd.read_csv(filepath)[["name", "ingredients", "cuisine", "time"]]
    df.dropna(inplace=True)
    df["name"]        = df["name"].str.lower().str.strip()
    df["ingredients"] = df["ingredients"].str.lower().str.strip()
    df["cuisine"]     = df["cuisine"].str.lower().str.strip()
    df.reset_index(drop=True, inplace=True)

    print(f"[INFO] Loaded {len(df)} recipes | {df['cuisine'].nunique()} cuisines")
    return df


# =========================================================
# 2. FEATURE ENGINEERING
#    TF-IDF + One-Hot Encoding + MinMax Normalization
# =========================================================
def create_features(df):
    # TF-IDF on ingredients (text vectorization)
    tfidf = TfidfVectorizer(max_features=200)
    ing_matrix = tfidf.fit_transform(df["ingredients"])

    # One-Hot Encoding for cuisine
    ohe = OneHotEncoder(handle_unknown="ignore")
    cuisine_matrix = ohe.fit_transform(df[["cuisine"]])

    # MinMax Normalization for cooking time
    scaler = MinMaxScaler()
    time_scaled = scaler.fit_transform(df[["time"]])
    time_sparse  = csr_matrix(time_scaled)

    # Combine all into one feature matrix
    features = hstack([ing_matrix, cuisine_matrix, time_sparse])
    print(f"[INFO] Feature matrix: {features.shape}")
    return features, tfidf, ohe, scaler


# =========================================================
# 3. PCA — Dimensionality Reduction
# =========================================================
def apply_pca(features):
    dense = features.toarray()
    n_comp = min(50, dense.shape[0], dense.shape[1])
    pca = PCA(n_components=n_comp, random_state=42)
    reduced = pca.fit_transform(dense)
    var = sum(pca.explained_variance_ratio_) * 100
    print(f"[INFO] PCA: {n_comp} components, {var:.1f}% variance retained")
    return reduced, pca


# =========================================================
# 4. KMeans CLUSTERING
# =========================================================
def apply_kmeans(reduced, df):
    n_clusters = df["cuisine"].nunique()   # driven by data, not hardcoded
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df = df.copy()
    df["cluster"] = kmeans.fit_predict(reduced)
    print(f"[INFO] KMeans: {n_clusters} clusters")
    return df, kmeans


# =========================================================
# 5. KNN CLASSIFIER — Predict Cuisine + Cross-Val Accuracy
# =========================================================
def train_knn(reduced, df):
    le = LabelEncoder()
    y  = le.fit_transform(df["cuisine"])

    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(reduced, y)

    # Cross-validation accuracy (5-fold)
    scores = cross_val_score(knn, reduced, y, cv=5, scoring="accuracy")
    print(f"[INFO] KNN 5-Fold Accuracy: {scores.mean()*100:.2f}% +/- {scores.std()*100:.2f}%")

    return knn, le


# =========================================================
# 6. SIMILARITY MATRICES
#    Cosine Similarity + Euclidean Distance
# =========================================================
def compute_similarity(features):
    cos_sim  = cosine_similarity(features)
    euc_dist = euclidean_distances(features)
    return cos_sim, euc_dist


# =========================================================
# 7. RECOMMEND BY RECIPE NAME
# =========================================================
def recommend_by_name(name, df, cos_sim, euc_dist):
    name = name.lower().strip()
    matches = df[df["name"] == name]

    if matches.empty:
        partial = df[df["name"].str.contains(name, na=False)]
        if partial.empty:
            print(f"[!] Recipe '{name}' not found.")
            return
        idx = partial.index[0]
        print(f"[INFO] Showing results for: '{df.iloc[idx]['name']}'")
    else:
        idx = matches.index[0]

    cos_scores = sorted(enumerate(cos_sim[idx]),  key=lambda x: x[1], reverse=True)[1:6]
    euc_scores = sorted(enumerate(euc_dist[idx]), key=lambda x: x[1])[1:6]

    print(f"\n=== Top 5 by Cosine Similarity ===")
    for i, score in cos_scores:
        r = df.iloc[i]
        print(f"  {r['name']:<30} cuisine: {r['cuisine']:<12} score: {score:.4f}")

    print(f"\n=== Top 5 by Euclidean Distance ===")
    for i, dist in euc_scores:
        r = df.iloc[i]
        print(f"  {r['name']:<30} cuisine: {r['cuisine']:<12} dist:  {dist:.4f}")


# =========================================================
# 8. CUSTOM RECOMMENDATION (ingredients + cuisine + time)
# =========================================================
def recommend_custom(ingredients, cuisine, time,
                     df, features, tfidf, ohe, scaler,
                     knn, le, pca, kmeans):

    ing_vec     = tfidf.transform([ingredients.lower()])
    cuisine_vec = ohe.transform([[cuisine.lower()]])
    time_vec    = scaler.transform([[float(time)]])

    query = hstack([ing_vec, cuisine_vec, csr_matrix(time_vec)])

    # Cosine similarity against all recipes
    scores = cosine_similarity(query, features).flatten()
    top    = scores.argsort()[::-1][:5]

    # PCA transform for KNN + KMeans
    query_pca    = pca.transform(query.toarray())
    pred_cuisine = le.inverse_transform(knn.predict(query_pca))[0]
    cluster      = kmeans.predict(query_pca)[0]

    print(f"\n=== Custom Recommendation ===")
    print(f"  Predicted Cuisine : {pred_cuisine}")
    print(f"  Cluster Group     : {cluster}")
    print(f"\n  Top 5 Matches:")
    for i in top:
        r = df.iloc[i]
        print(f"  {r['name']:<30} cuisine: {r['cuisine']:<12} score: {scores[i]:.4f}")


# =========================================================
# MAIN
# =========================================================
def main():
    df = load_data("ml_recipes.csv")

    features, tfidf, ohe, scaler = create_features(df)
    reduced, pca                 = apply_pca(features)
    df, kmeans                   = apply_kmeans(reduced, df)
    knn, le                      = train_knn(reduced, df)
    cos_sim, euc_dist            = compute_similarity(features)

    print("\n" + "="*50)
    print("   RECIPE RECOMMENDATION SYSTEM READY")
    print("="*50)

    while True:
        print("\n  1. Recommend by recipe name")
        print("  2. Custom recommendation (ingredients + cuisine + time)")
        print("  3. Exit")
        ch = input("\nEnter choice: ").strip()

        if ch == "1":
            name = input("Enter recipe name: ")
            recommend_by_name(name, df, cos_sim, euc_dist)

        elif ch == "2":
            ing   = input("Enter ingredients (space separated): ")
            cuis  = input("Enter cuisine (e.g. indian, italian): ")
            time  = input("Enter cooking time (minutes): ")
            recommend_custom(ing, cuis, time,
                             df, features, tfidf, ohe, scaler,
                             knn, le, pca, kmeans)

        elif ch == "3":
            print("Goodbye!")
            break
        else:
            print("[!] Invalid choice. Enter 1, 2, or 3.")


if __name__ == "__main__":
    main()